In [1]:
#A

import numpy as np

X=np.random.multivariate_normal(np.zeros(20),np.eye(20),size=100)

i=np.random.choice(np.arange(1,21),size=4,replace=False)

i1,i2,i3,i4=i[0],i[1],i[2],i[3]

A=np.random.normal(0,0.25**(1/2),size=4)

a,b,c,d=A[0],A[1],A[2],A[3]

nk=np.random.normal(0,0.0001,size=100)

Y=[]

for k in range(1,101):
    y=a*X[k-1,i1-1]+b*X[k-1,i2-1]+c*X[k-1,i3-1]+d*X[k-1,i4-1]+nk[k-1]
    Y.append(y)

print(Y[:5])

[np.float64(-0.4189974395145196), np.float64(-1.0739413857305402), np.float64(1.0447677708283887), np.float64(0.1893754227917104), np.float64(1.3899183248950095)]


In [2]:
#B

beta_ols=np.linalg.inv(X.T@X)@X.T@Y

print(beta_ols)

from sklearn.linear_model import LinearRegression

model = LinearRegression(fit_intercept=False)   #because we dont have any intercept
model.fit(X,Y)                                  #This internally computes OLS solution.
beta_sklearn = model.coef_

print(beta_sklearn)

[ 1.30233252e-05 -5.05421728e-06  1.06771116e+00  7.33582334e-06
 -1.41787690e-05 -4.56229812e-06 -7.06305699e-06 -3.20502025e-05
  3.60404255e-06  3.22605431e-02 -1.83637807e-05 -1.06832333e-05
 -6.30377151e-06 -1.18545842e-05 -1.00491611e-01 -1.00087204e-05
  9.95179544e-01  1.29191489e-05 -7.17990528e-06  1.47740575e-05]
[ 1.30233252e-05 -5.05421729e-06  1.06771116e+00  7.33582334e-06
 -1.41787690e-05 -4.56229812e-06 -7.06305699e-06 -3.20502025e-05
  3.60404255e-06  3.22605431e-02 -1.83637807e-05 -1.06832333e-05
 -6.30377152e-06 -1.18545842e-05 -1.00491611e-01 -1.00087204e-05
  9.95179544e-01  1.29191489e-05 -7.17990528e-06  1.47740575e-05]


In [3]:
#C

p=X.shape[1]
beta_ridge=np.linalg.inv(X.T@X+20*np.eye(p))@X.T@Y

print(beta_ridge)

from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=20, fit_intercept=False)
ridge_model.fit(X,Y)
beta_sklearn = ridge_model.coef_

print(beta_sklearn)

top5 = np.argsort(np.abs(beta_ridge))[::-1][:5]    #np.argsort---Returns indices in increasing order.
                                                   # [::-1][:5] ----reverse the order
print(top5+1)


[ 0.03309348 -0.00313105  0.88993302  0.0281916   0.02980659 -0.01281671
 -0.0381276  -0.01386865  0.01530678  0.00452875 -0.00477833 -0.00885568
  0.01391901  0.01836652 -0.05401068 -0.02772207  0.82191314 -0.04198756
  0.01787812  0.02442866]
[ 0.03309348 -0.00313105  0.88993302  0.0281916   0.02980659 -0.01281671
 -0.0381276  -0.01386865  0.01530678  0.00452875 -0.00477833 -0.00885568
  0.01391901  0.01836652 -0.05401068 -0.02772207  0.82191314 -0.04198756
  0.01787812  0.02442866]
[ 3 17 15 18  7]


In [4]:
# D

from sklearn.linear_model import Lasso

lasso_model = Lasso(alpha=20, fit_intercept=False, max_iter=10000)
lasso_model.fit(X, Y)

beta_lasso = lasso_model.coef_

print(beta_lasso)

top5 = np.argsort(np.abs(beta_lasso))[::-1][:5]

print(top5 + 1)

[ 0. -0.  0.  0.  0. -0. -0. -0.  0. -0. -0. -0.  0.  0.  0. -0.  0. -0.
  0.  0.]
[20 19  2  3  4]


In [5]:
# =======================
# (e) 5-Fold Cross Validation for Ridge and Lasso
# =======================

import numpy as np
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error

alphas = [1, 2, 5, 10, 20, 50, 100, 1000, 10000]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# RIDGE CROSS VALIDATION 

ridge_cv_errors = []

for alpha in alphas:
    fold_errors = []

    for train_idx, val_idx in kf.split(X):

        X_train, X_val = X[train_idx], X[val_idx]
        Y_train, Y_val = np.array(Y)[train_idx], np.array(Y)[val_idx]

        p = X_train.shape[1]

        # Closed form Ridge
        beta = np.linalg.inv(X_train.T @ X_train + alpha*np.eye(p)) @ X_train.T @ Y_train

        Y_pred = X_val @ beta

        mse = np.mean((Y_val - Y_pred)**2)
        fold_errors.append(mse)

    avg_error = np.mean(fold_errors)
    ridge_cv_errors.append(avg_error)

print("Ridge CV Errors:")
for a, err in zip(alphas, ridge_cv_errors):
    print(f"alpha={a}, error={err:.6f}")

best_alpha_ridge = alphas[np.argmin(ridge_cv_errors)]
print("\nBest alpha for Ridge (scratch) =", best_alpha_ridge)

# =====================================================
# RIDGE USING GridSearchCV
# =====================================================

ridge_grid = GridSearchCV(
    Ridge(fit_intercept=False),
    param_grid={"alpha": alphas},
    cv=5,
    scoring="neg_mean_squared_error"
)

ridge_grid.fit(X, Y)

print("Best alpha for Ridge (GridSearchCV) =", ridge_grid.best_params_["alpha"])
print("Best CV Error =", -ridge_grid.best_score_)

# =====================================================
# LASSO CROSS VALIDATION FROM SCRATCH
# =====================================================

lasso_cv_errors = []

for alpha in alphas:
    fold_errors = []

    for train_idx, val_idx in kf.split(X):

        X_train, X_val = X[train_idx], X[val_idx]
        Y_train, Y_val = np.array(Y)[train_idx], np.array(Y)[val_idx]

        model = Lasso(alpha=alpha, fit_intercept=False, max_iter=10000)
        model.fit(X_train, Y_train)

        Y_pred = model.predict(X_val)

        mse = np.mean((Y_val - Y_pred)**2)
        fold_errors.append(mse)

    avg_error = np.mean(fold_errors)
    lasso_cv_errors.append(avg_error)

print("\nLasso CV Errors:")
for a, err in zip(alphas, lasso_cv_errors):
    print(f"alpha={a}, error={err:.6f}")

best_alpha_lasso = alphas[np.argmin(lasso_cv_errors)]
print("\nBest alpha for Lasso (scratch) =", best_alpha_lasso)

# =====================================================
# LASSO USING GridSearchCV
# =====================================================

lasso_grid = GridSearchCV(
    Lasso(fit_intercept=False, max_iter=10000),
    param_grid={"alpha": alphas},
    cv=5,
    scoring="neg_mean_squared_error"
)

lasso_grid.fit(X, Y)

print("Best alpha for Lasso (GridSearchCV) =", lasso_grid.best_params_["alpha"])
print("Best CV Error =", -lasso_grid.best_score_)

Ridge CV Errors:
alpha=1, error=0.000588
alpha=2, error=0.002248
alpha=5, error=0.012368
alpha=10, error=0.040949
alpha=20, error=0.119503
alpha=50, error=0.381341
alpha=100, error=0.739588
alpha=1000, error=2.058831
alpha=10000, error=2.415112

Best alpha for Ridge (scratch) = 1
Best alpha for Ridge (GridSearchCV) = 1
Best CV Error = 0.0006242592713854884

Lasso CV Errors:
alpha=1, error=1.804576
alpha=2, error=2.461370
alpha=5, error=2.461370
alpha=10, error=2.461370
alpha=20, error=2.461370
alpha=50, error=2.461370
alpha=100, error=2.461370
alpha=1000, error=2.461370
alpha=10000, error=2.461370

Best alpha for Lasso (scratch) = 1
Best alpha for Lasso (GridSearchCV) = 1
Best CV Error = 1.8324276890201439


In [6]:
help(GridSearchCV)

Help on class GridSearchCV in module sklearn.model_selection._search:

class GridSearchCV(BaseSearchCV)
 |  GridSearchCV(
 |      estimator,
 |      param_grid,
 |      *,
 |      scoring=None,
 |      n_jobs=None,
 |      refit=True,
 |      cv=None,
 |      verbose=0,
 |      pre_dispatch='2*n_jobs',
 |      error_score=nan,
 |      return_train_score=False
 |  )
 |
 |  Exhaustive search over specified parameter values for an estimator.
 |
 |  Important members are fit, predict.
 |
 |  GridSearchCV implements a "fit" and a "score" method.
 |  It also implements "score_samples", "predict", "predict_proba",
 |  "decision_function", "transform" and "inverse_transform" if they are
 |  implemented in the estimator used.
 |
 |  The parameters of the estimator used to apply these methods are optimized
 |  by cross-validated grid-search over a parameter grid.
 |
 |  Read more in the :ref:`User Guide <grid_search>`.
 |
 |  Parameters
 |  ----------
 |  estimator : estimator object
 |      Thi